# Time-Series Trend and Rolling Metrics Analysis

Analysing temporal patterns in demand, operational behaviour, and rider experience.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from roadies.ingestion.loaders import load_csv
from roadies.features.demand_supply import engineer_demand_supply_features
from roadies.features.surge import engineer_surge_features
from roadies.features.acceptance import engineer_acceptance_features
from roadies.features.cancellation import engineer_cancellation_features
from roadies.features.experience import engineer_experience_features
from roadies.features.demand_period import classify_high_demand
from roadies.analysis.time_series import (
    aggregate_time_series,
    calculate_rolling_metrics,
    analyze_city_time_series,
    compare_high_demand_time_series,
    analyze_temporal_dimensions,
)

In [ ]:
# Load and engineer features
df = load_csv(Path('data/raw/rides.csv'))
df, _ = engineer_demand_supply_features(df)
df, _ = engineer_surge_features(df)
df, _ = engineer_acceptance_features(df)
df, _ = engineer_cancellation_features(df)
df, _ = engineer_experience_features(df)
df, _ = classify_high_demand(df)
print(f'Dataset: {len(df)} rows')

In [ ]:
# Daily trend
daily = aggregate_time_series(df, grain='day')
px.line(daily, x='request_timestamp_day', y='ride_id_count', title='Daily Ride Volume')

In [ ]:
# Rolling metrics
rolling = calculate_rolling_metrics(df, grain='day', window=7)
px.line(rolling, x='request_timestamp_day', y=['rider_cancelled_mean', 'rider_cancelled_mean_rolling_7'], title='7-Day Rolling Cancellation')

In [ ]:
# City comparison
city_ts = analyze_city_time_series(df)
px.line(city_ts, x='request_timestamp_day', y='wait_time_minutes_mean', color='city', title='Daily Wait Time by City')

In [ ]:
# High-demand vs normal
high, normal = compare_high_demand_time_series(df)
fig = go.Figure()
fig.add_trace(go.Scatter(x=high['request_timestamp_day'], y=high['surge_multiplier_mean'], name='High Demand'))
fig.add_trace(go.Scatter(x=normal['request_timestamp_day'], y=normal['surge_multiplier_mean'], name='Normal'))
fig.update_layout(title='Surge: High Demand vs Normal')
fig.show()

In [ ]:
# Temporal dimensions
dims = analyze_temporal_dimensions(df)
for name, dim_df in dims.items():
    print(f'\n{name}:')
    print(dim_df.to_string())